# AIOps 實戰課程 — 全局架構總覽

本 lab 是整套課程的起點，也是課程的地圖。在你動手跑任何一行程式之前，先看清楚整張 Pipeline。

圖表也可在 原始 draw.io 來源在 `project/202608-aia-network-monitoring-aiops/diagrams/aiops_pipeline_overview.drawio` 以 draw.io 開啟編輯。


In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/self-study/diagrams/aiops_pipeline_overview.svg"))

## 本 Lab 的學習目標

**本 lab 涵蓋 Pipeline 的採集層（第一個方框）。**
先讓資料正確流進 Prometheus，後面所有的偵測、預測、RCA 才有原料。

### 學習目標

1. 理解 Prometheus pull model：scrape 是什麼，scrape interval 的取捨
2. 驗證採集鏈（node_exporter → Prometheus → Grafana）是否正常
3. 查詢你的 PC 網路介面清單與即時流量
4. 掌握 PromQL 的四種查詢模式

### 前置條件

- node_exporter 已安裝並啟動（`http://localhost:9100/metrics` 有回應）
- Prometheus 已啟動（`http://localhost:9090`）
- Grafana Local 已啟動（`http://localhost:3000`，設定步驟見 [03a-install-grafana-local.md](../../getting-started/03a-install-grafana-local.md)）

## 設計主線：從自己的機器開始理解 production monitoring

本章用本機 node_exporter 或 windows_exporter 當作第一個監控目標。這不是玩具例子，而是縮小版的 production pattern：endpoint 暴露 metrics，Prometheus 定期抓取，Grafana 顯示狀態。

請把每個查詢都連回實務設計：

1. **指標來源是否可信**：服務是否真的 up？metrics 是否更新？label 是否能辨識設備與介面？
2. **PromQL 是否回答問題**：rate、sum by、label filter 分別解決不同問題，不要把所有查詢都寫成單一 metric lookup。
3. **dashboard 是否支援決策**：圖表應該讓工程師知道是否需要行動，而不是只顯示數字。

設計原則：可觀測性不是裝工具，而是建立可被驗證的資料流。


---

## 這條 Pipeline 在解決什麼問題

一個中型網路環境有數百個設備、每個設備有數十個埠、每個埠有十幾個指標，每 5 分鐘更新一次。
手動監控在這個規模下已經不可行——不是因為人不夠努力，而是因為資訊量超過人腦的注意力上限。

AIOps 的角色不是取代工程師的判斷，而是把**可以自動化的感知工作**交給機器：

| 工作 | 人類做的成本 | 機器替代後 |
|---|---|---|
| 看所有指標有沒有異常 | 需要持續注意，不可能 24/7 | 自動偵測，秒級 |
| 計算 z-score、管制線 | 需要統計背景、手動計算 | PromQL 或 Python 一行 |
| 比較不同 port 的同步異常 | 需要開多個視窗對比 | ML 跨維度自動比對 |
| 生成 RCA 報告 | 需要整理多個資訊來源 | LLM 結構化輸出 |

**機器感知之後，人類做什麼？**

判斷、決策、行動。機器說「這個 port 的 z-score 是 4.3」，人類說「這是計畫中的備份視窗，不是異常」。
這個決策沒有辦法自動化——因為它需要業務 context，而業務 context 在你的腦袋裡，不在指標裡。

這就是為什麼每個 lab 都有「⚠ 人類驗證點」。不是形式，是流程的關鍵節點。


---

## 課程全 Pipeline — 各階段角色與取捨

### Stage 1：採集層（Lab 00）

**解決的問題：** 指標散落在不同設備，格式不一致，沒有統一的查詢介面。

Prometheus 的 pull model 讓你用一套 PromQL 查詢所有來源。
node_exporter 是最直接的 OS-level 採集方式；企業環境則可能是 SNMP exporter 或 RRD 轉接。

| 取捨 | 說明 |
|---|---|
| Scrape interval 短 → 資料更細緻 | 但 Prometheus 儲存成本增加，適合高頻異常偵測 |
| Scrape interval 長 → 儲存省 | 但change point detection的精確度下降，適合趨勢分析 |
| Pull model | Prometheus 主動抓，設備不需要知道 Prometheus 的存在；代價是需要在 Prometheus 維護目標清單 |

---

### Stage 2：特徵工程（Lab 01）

**解決的問題：** 原始 Counter 值（累積位元組數）本身沒有偵測意義。

`node_network_receive_bytes_total` 只會一直增加——你需要 `rate()` 才能看到「速率」。
更進一步，`(tx_bytes + rx_bytes)` 是總流量，`rx_errors / rx_packets` 是錯誤率；
這些組合特徵才是異常偵測的輸入。

```
Raw metrics           →  Features
Counter               →  rate()  (bytes/sec increment)
bytes + bytes         →  traffic_total  (total throughput)
errors / packets      →  error_rate  (quality indicator)
rolling 12-point std  →  baseline_std  (dynamic baseline)
lag_12                →  autocorrelation feature  (periodic pattern)
```

**特徵工程的決策沒有標準答案。** 你選的特徵決定了後面三個偵測層能看到什麼。

---

### Stage 3：偵測層（Labs 02 / 03 / 04）— 三個層次，互補而非替代

三種方法的存在理由不同，適合抓到的異常類型也不同：

```
Anomaly type                          → Suitable method
───────────────────────────────────────────────────────────────
Single-metric spike (obvious)         → Lab 02 statistical baseline (z-score, P95)
Slow trend drift                      → Lab 03 SPC (CUSUM is drift-sensitive)
Multi-metric subtle co-movement       → Lab 04 ML (Isolation Forest, cross-dimensional)
Time-series structural break          → Lab 02 change point detection (dual MA divergence)
```

| | Lab 02 統計基線 | Lab 03 SPC | Lab 04 ML |
|---|---|---|---|
| 可解釋性 | 高（公式透明）| 高 | 中（黑盒成分）|
| 需要訓練資料 | 否 | 否 | 是（歷史正常資料）|
| 多維偵測 | 否 | 否 | 是 |
| 計算成本 | 低 | 低 | 中～高 |
| 部署複雜度 | 低（PromQL）| 低 | 高（ML sidecar 服務）|
| 誤報率 | 較高（單維度）| 中 | 較低（多維互補）|

**為什麼要三層都學？** 因為沒有任何單一方法能抓到所有異常。
先部署 Lab 02 作為第一道防線；Lab 03 補抓製程漂移；Lab 04 補抓複合型異常。

---

### Stage 4：告警降噪（Lab 05）

**解決的問題：** 三層偵測 × 多個 port × 多個指標 = 告警爆炸。

告警疲勞（alert fatigue）是 AIOps 最常見的失敗原因：
告警太多 → 工程師開始忽略告警 → 真正的問題被淹沒 → 比沒有告警更危險。

Lab 05 的目標是讓每一個送到工程師的告警都有人去看。

| 降噪技術 | 解決的場景 |
|---|---|
| Time視窗合併 | 同一個 port 連續觸發的告警合成一筆 |
| 跨 port 關聯 | 多個 port 同時 spike → 可能是上游問題，合成一筆 |
| 抑制規則 | 父設備異常時，子設備的告警自動靜音 |
| 告警去重複 | 同類型告警在冷卻時間內只送一次 |

---

### Stage 5：預測層（Lab 06）

**解決的問題：** 偵測是被動的（問題已發生才知道）；預測讓你主動行動。

SARIMA 適合有明顯週期性的流量（日週期、周週期）；
LSTM 適合非線性、長依賴的序列，但需要更多訓練資料和計算資源。

預測的輸出不是「會不會異常」，而是「正常情況下應該是多少」——
讓偵測層可以問：「實際值偏離預測值多少個標準差？」這比純靜態基線更準確。

---

### Stage 6：根因分析（Labs 07 / 08）

**解決的問題：** 偵測告訴你「什麼時候、哪裡」出問題；RCA 告訴你「為什麼、怎麼修」。

| | Lab 07 LLM 輔助 RCA | Lab 08 Agentic AI RCA |
|---|---|---|
| 互動方式 | 單次 LLM 呼叫（prompt → response）| 多輪工具循環（tool_use loop）|
| 能力 | 生成根因候選清單 | 主動查詢額外資訊後再診斷 |
| 自主度 | 低（你提供上下文）| 較高（Agent 自己決定查什麼）|
| 適用場景 | 異常上下文已整理好 | 需要跨多個資料來源動態查詢 |
| 人類驗證 | 評估根因清單是否合理 | 確認 Agent 行動計畫後才執行 |

LLM 不會知道你的業務 context——你需要在 prompt 裡告訴它「這個 port 連接的是什麼設備、上週有沒有變更」。
這個上下文建構（context engineering）是 RCA 品質的決定因素，不是 LLM 本身。

---

### Stage 7：部署與閉環（Lab 08 生產部署）

驗證通過的偵測邏輯部署到 Prometheus alert rules；
LLM webhook 自動接收告警、生成 RCA 摘要、推送到 Slack / PagerDuty。

這個環節把前六個 stage 的輸出串成一個持續運行的系統。


---

## 這套課程的立場：人類驗證，AI 加速

上面六個 stage 的每一個，AI 都比人類快、比人類不疲勞。
但有一類工作 AI 無法替代：**用業務 context 做判斷**。

| AI 做的事 | 人類必須做的事 |
|---|---|
| 計算 z-score、管制線、ML 異常分數 | 判斷這個分數代表真正的問題還是計畫中的事件 |
| 跨 port 比較，找出同步異常 | 提供背景：這兩個 port 是否在同一個上游 |
| 生成 RCA 候選清單 | 評估哪個候選根因在你的環境裡成立 |
| 執行 Agentic 工具循環 | 確認 Agent 的行動計畫後才授權執行 |

**每個 lab 的「⚠ 人類驗證點」標記的是 pipeline 裡人類判斷不可或缺的節點。**
不是形式、不是可以跳過的步驟——是整個系統設計裡最重要的環節。


---
**概念說明 ▸ 講師引導** — 請聆聽講師說明 Prometheus 架構，完成後再執行以下 cell。

---

## 概念：Prometheus Pull Model 與 node_exporter

### 系統架構圖

（見下方圖表）

---

### Pull Model 運作原理

Prometheus 採用 **主動抓取（pull）** 模式，與傳統 push 架構相反：

```
Push architecture (traditional)      Pull architecture (Prometheus)
────────────────────────────         ─────────────────────────────────
Device → pushes data → monitor        Prometheus → pulls data ← device waits
(device must know where monitor is)   (device only needs to expose /metrics)

Drawbacks:                            Advantages:
  Complex device config                 Simple device config (just listen)
  Data lost when monitor is down        Prometheus controls scrape rate
  Device may send wrong format          Unified retry, timeout, label injection
```

每隔 `scrape_interval`（預設 15 秒），Prometheus 向每個 target 的 `/metrics` 發出 HTTP GET，解析 Exposition Format，寫入本地 TSDB。

---

### node_exporter 暴露哪些指標？

```
node_exporter metric classification
│
├── network
│   ├── node_network_receive_bytes_total    ← received bytes (cumulative counter)
│   ├── node_network_transmit_bytes_total   ← transmitted bytes (cumulative counter)
│   ├── node_network_receive_packets_total  ← received packet count
│   ├── node_network_receive_errs_total     ← receive error count
│   └── node_network_receive_drop_total     ← discarded packet count
│
├── CPU
│   └── node_cpu_seconds_total{mode="idle|user|system"}
│
├── memory
│   ├── node_memory_MemTotal_bytes
│   └── node_memory_MemFree_bytes
│
└── disk
    └── node_disk_read_bytes_total
```

---

### Counter vs Gauge — 關鍵區別

```
Counter (cumulative)                      Gauge (current value)

  value ↑  monotonically increasing           value ↑  can rise or fall
     (B)│        ┌──────────                     (B)│   ╱╲    ╱╲
        │  ──────┘                                  │  ╱  ╲  ╱  ╲
        └───────────────→ Time                      └──────────────→ Time

  requires rate() to get throughput          use directly
  rate(bytes_total[1m]) = bytes/sec          memory_free_bytes = available memory now

  resets to 0 on restart → rate() compensates automatically  restores true value after restart
```

**為什麼 rate() 要指定 `[1m]`？**

`rate(metric[1m])` 的 `[1m]` 是**回溯窗口（lookback 視窗）**，
告訴 Prometheus「用過去 1 分鐘的樣本計算斜率」。
窗口越長，曲線越平滑，短暫突刺被稀釋；窗口越短，越靈敏，噪音也越大。
AIOps 實務上常用 `[1m]` 做即時偵測，`[5m]` 做趨勢分析。

---

### TSDB 儲存結構（簡介）

Prometheus 內建時序資料庫（TSDB），每個時序用**標籤組合**唯一識別：

```
metric_name{label1="v1", label2="v2"} → independent time series

node_network_receive_bytes_total{device="en0", instance="localhost:9100", job="node-exporter"}
node_network_receive_bytes_total{device="lo",  instance="localhost:9100", job="node-exporter"}
─── two distinct device labels = two independent time series ───
```

**標籤設計原則：** 高基數（cardinality）標籤（例如用 UUID 當標籤值）會讓 TSDB 產生海量時序，拖垮查詢效能。實務上避免把 request_id、user_id 這類唯一值放進標籤。

---

**參考資料：**
- Prometheus pull model 說明：https://prometheus.io/docs/introduction/faq/#why-do-you-pull-rather-than-push
- node_exporter 指標列表：https://github.com/prometheus/node_exporter#enabled-by-default


In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/workshop/diagrams/lab00_observability_stack_arch.svg"))

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 0：環境準備 — 匯入套件與定義工具函數

In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 設定繪圖風格
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

PROM = "http://localhost:9090"

def prom_range(expr, hours=6, step="15s"):
    """查詢 Prometheus range query API，回傳原始 JSON。"""
    end = datetime.utcnow()
    start = end - timedelta(hours=hours)
    params = urllib.parse.urlencode({
        "query": expr,
        "start": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "end":   end.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "step":  step,
    })
    url = f"{PROM}/api/v1/query_range?{params}"
    with urllib.request.urlopen(url, timeout=8) as r:
        return json.loads(r.read())

def to_df(result, value_col="value"):
    """將 Prometheus range query 結果轉換為 pandas DataFrame。"""
    rows = []
    for series in result["data"]["result"]:
        dev = series["metric"].get("device",
              series["metric"].get("instance", "?"))
        for ts, val in series["values"]:
            rows.append({
                "timestamp": pd.Timestamp(ts, unit="s"),
                "device": dev,
                value_col: float(val)
            })
    return pd.DataFrame(rows)

print("✅ 工具函數載入完成")
print(f"Prometheus 目標：{PROM}")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1：驗證服務狀態

在執行任何查詢之前，先確認三個服務都在線：

| 服務 | 預設埠 | 功能 |
|------|--------|------|
| Prometheus | 9090 | 時序資料庫 + 查詢引擎 |
| Grafana Local | 3000 | 視覺化儀表板（Prometheus 資料源）|
| node_exporter | 9100 | OS 指標採集代理 |

三個服務缺一不可：node_exporter 提供資料來源，Prometheus 抓取並儲存，Grafana Local 視覺化查詢結果。Grafana Cloud（選用）可額外設定 remote_write 把指標同步到雲端，但不是本 lab 的必要條件。


In [ ]:
# 依序檢查三個服務是否正常運行
checks = [
    ("http://localhost:9090/-/healthy",  "Prometheus  :9090"),
    ("http://localhost:9100/metrics",    "node_exporter :9100"),
]

all_ok = True
for url, label in checks:
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            status = r.status
        print(f"✅ {label}  (HTTP {status})")
    except Exception as e:
        print(f"❌ {label}  — 無法連線：{e}")
        all_ok = False

print()
if all_ok:
    print("🎉 所有服務正常！可以繼續下一步。")
else:
    print("⚠ 有服務未啟動。請確認 exporter、Prometheus、Grafana、node_exporter 已依 getting-started 指南啟動。")
    print("  或向講師確認安裝步驟。")

> **如果服務無法連線：**
> 1. 確認 node_exporter 正在執行：`http://localhost:9100/metrics` 有回應
> 2. 確認 Prometheus 可開啟：`http://localhost:9090`
> 3. 確認 Grafana Local 已啟動：`http://localhost:3000`
> 4. 稍等 30 秒後重試

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2：查詢你的 PC 網路介面清單

在做流量分析之前，先確認 Prometheus 能看到哪些網路介面，以及哪些值得監控。

**為什麼要過濾介面？**
`node_exporter` 暴露所有介面，包括虛擬介面（loopback、Docker bridge、VPN 隧道）。
監控虛擬介面的流量對網路異常偵測沒有意義，反而會引入噪音。

**常見需要排除的介面（依環境不同）：**
```
lo, lo0          → loopback (local loopback)
docker0          → Docker bridge
veth*            → Docker container virtual interface
br-*             → Docker custom bridge network
utun*, tun*      → VPN tunnel (macOS/Linux)
```


In [ ]:
try:
    result = prom_range('node_network_receive_bytes_total', hours=0.1, step="60s")
    all_devices = sorted({s["metric"].get("device", "?") 
                          for s in result["data"]["result"]})
    print(f"Prometheus 找到 {len(all_devices)} 個網路介面：")
    print()
    for dev in all_devices:
        # 判斷介面類型
        tag = ""
        if dev == "lo":           tag = "  ← 迴路介面（loopback），通常忽略）"
        elif dev.startswith("eth") or dev.startswith("en"): tag = "  ← 有線網路"
        elif dev.startswith("wl") or dev.startswith("wlan"): tag = "  ← 無線網路"
        elif dev.startswith("docker") or dev.startswith("veth"): tag = "  ← Docker 虛擬介面"
        print(f"  {dev}{tag}")
    
    # 過濾出「真實」介面
    real_devices = [d for d in all_devices 
                    if not d.startswith(("lo", "docker", "veth", "br-", "vbox"))]
    print()
    print(f"建議監控的真實介面：{real_devices}")
    MAIN_DEVICE = real_devices[0] if real_devices else all_devices[0]
    print(f"本 Lab 主要使用介面：{MAIN_DEVICE}")

except Exception as e:
    print(f"⚠ Prometheus 無法連線（{e}）— 使用示範裝置名稱")
    MAIN_DEVICE = "eth0"
    real_devices = ["eth0"]
    print(f"示範裝置：{MAIN_DEVICE}")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3：繪製即時接收速率圖（最近 30 分鐘）

**這一步做什麼？**

從 Prometheus 拉取你的 PC 網路介面的接收速率，繪製時序圖。
若 Prometheus 不可用，會自動 fallback 到合成資料，確保你能繼續學習。

**流量速率的計算（PromQL 等效）：**
```promql
rate(node_network_receive_bytes_total{device="en0"}[1m])
```
這個查詢返回「過去 1 分鐘平均每秒接收多少 bytes」。
Python 側是用 pandas 對 counter 差值 ÷ 秒數來重現相同計算。

**單位換算：**
```
bytes/sec → KB/s:  ÷ 1,024
bytes/sec → MB/s:  ÷ 1,048,576
bytes/sec → Mbps:  ÷ 131,072  (network-standard Mbps = megabits/sec)
```
本 Lab 使用 bytes/sec（raw），視覺化時可依需求換算。


In [ ]:
import numpy as np

def generate_synthetic_traffic(hours=0.5, step_seconds=15):
    """當 Prometheus 不可用時，產生示範用合成Traffic資料。"""
    n = int(hours * 3600 / step_seconds)
    end = datetime.utcnow()
    timestamps = [end - timedelta(seconds=step_seconds * (n - i)) for i in range(n)]
    # 模擬正常Traffic + 兩個突刺
    base = 1_000_000  # 1 MB/s 基準
    noise = np.random.normal(0, 100_000, n)
    signal = np.full(n, base) + noise
    # 加入突刺
    spike1 = int(n * 0.4)
    signal[spike1:spike1+3] += 5_000_000
    spike2 = int(n * 0.75)
    signal[spike2:spike2+2] += 8_000_000
    return pd.DataFrame({
        "timestamp": [pd.Timestamp(t) for t in timestamps],
        "device": "eth0",
        "rx_rate": np.maximum(signal, 0)
    })

# 查詢receive rate
filter_expr = 'device!~"lo|docker.*|veth.*|br-.*"'
rx_expr = f'rate(node_network_receive_bytes_total{{{filter_expr}}}[1m])'

try:
    result = prom_range(rx_expr, hours=0.5, step="15s")
    df = to_df(result, value_col="rx_rate")
    print(f"✅ Prometheus：取得 {len(df)} 筆資料")
    print(f"   裝置：{df['device'].unique().tolist()}")
except Exception as e:
    print(f"⚠ Prometheus 無法連線（{e}）— 使用合成資料")
    df = generate_synthetic_traffic(hours=0.5)
    print(f"   合成資料：{len(df)} 筆")

print(f"\n資料時間範圍：")
print(f"  開始：{df['timestamp'].min()}")
print(f"  結束：{df['timestamp'].max()}")
df.head()

In [ ]:
# 繪製receive rate折線圖
fig, ax = plt.subplots(figsize=(12, 4))

for dev, grp in df.groupby("device"):
    grp = grp.sort_values("timestamp")
    ax.plot(grp["timestamp"], grp["rx_rate"] / 1e6,  # Convert to MB/s
            label=dev, linewidth=1.5)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_xlabel("Time")
ax.set_ylabel("Receive rate (MB/s)")
ax.set_title("Network receive rate: last 30 minutes")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n統計摘要（MB/s）：")
print(df.groupby("device")["rx_rate"].agg(['mean','std','max']).map(lambda x: f"{x/1e6:.3f}"))

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4：PromQL 基礎 — 4 種查詢模式

PromQL 是 Prometheus 的查詢語言，語法兼具向量運算和時序函數，初看有點陌生，但 4 種核心模式涵蓋日常 AIOps 工作的 80%。

---

### 模式 1：即時值（Instant Vector）

直接查詢指標名稱，取得每個時序的**當前值**：

```promql
node_memory_MemFree_bytes
```

回傳格式：`標籤組合 → 最新樣本值`

適用：Dashboard 快覽、告警條件（`> 閾值`）、Gauge 類型指標。

---

### 模式 2：rate() — Counter 轉速率

Counter 只增不減，必須用 `rate()` 換算成「每秒增量」才有意義：

```promql
rate(node_network_receive_bytes_total[1m])
```

等同計算：
```
(last sample in past 1 min − earliest sample) ÷ elapsed seconds
```

實務注意事項：
- 回溯窗口至少要涵蓋 2 個抓取點（若 scrape_interval=15s，`[30s]` 是最短合理窗口）
- Counter 重啟歸零（reset）時 rate() 會自動補償，不會產生負值

---

### 模式 3：sum by — 跨標籤聚合

把多條時序依標籤分組，再做加總：

```promql
sum(rate(node_network_receive_bytes_total{device!~"lo|docker.*|veth.*"}[1m])) by (device)
```

拆解讀法：
1. `{device!~"lo|docker.*|veth.*"}` — 正規表示式過濾，排除虛擬介面
2. `rate(...[1m])` — 每條時序轉速率
3. `sum(...) by (device)` — 依 device 標籤分組加總（保留 device，合併其他標籤）

常用聚合函數：`sum`、`avg`、`max`、`min`、`count`、`topk(5, ...)`

---

### 模式 4：label filter — 精確過濾

```promql
node_network_receive_bytes_total{device="en0"}
```

過濾運算子：
| 運算子 | 意義 | 範例 |
|--------|------|------|
| `=` | 完全相等 | `{job="node-exporter"}` |
| `!=` | 不等於 | `{device!="lo"}` |
| `=~` | 正規表示式匹配 | `{device=~"en.*"}` |
| `!~` | 正規表示式排除 | `{device!~"lo|docker.*"}` |

---

### 組合應用：偵測高 流量 介面

```promql
topk(3,
  rate(node_network_receive_bytes_total{device!~"lo|docker.*|veth.*"}[5m])
)
```

這一行 PromQL 的意思：從非虛擬介面中，找出過去 5 分鐘接收速率最高的 3 個介面。
這就是 AIOps 偵測的雛形：**從所有資料中自動找出最值得關注的對象**。


In [ ]:
# PromQL 範例 1：即時值（Gauge 類型）
# 查詢current的可用記憶體
try:
    r = prom_range('node_memory_MemFree_bytes', hours=0.1, step='60s')
    df1 = to_df(r, 'mem_free')
    latest = df1.sort_values('timestamp').iloc[-1]
    print(f"模式 1 — 即時值（Gauge）：")
    print(f"  可用記憶體：{latest['mem_free'] / 1e9:.2f} GB")
except Exception as e:
    print(f"模式 1 — 即時值：⚠ {e}（示範：目前可用記憶體 = 4.2 GB）")

In [ ]:
# PromQL 範例 2：rate() — Counter 轉速率
# rate() 計算每秒增加量，是使用 Counter 的標準方式
try:
    expr = 'rate(node_network_receive_bytes_total{device!~"lo|docker.*|veth.*"}[1m])'
    r = prom_range(expr, hours=0.5, step='15s')
    df2 = to_df(r, 'rx_rate')
    avg = df2['rx_rate'].mean()
    print(f"模式 2 — rate()（Counter → 速率）：")
    print(f"  過去 30 分鐘平均receive rate：{avg/1e6:.3f} MB/s")
    print(f"  PromQL：{expr}")
except Exception as e:
    print(f"模式 2 — rate()：⚠ {e}")

In [ ]:
# PromQL 範例 3：sum by — 跨標籤聚合
# 將所有非虛擬介面的Traffic加總
try:
    expr = 'sum(rate(node_network_receive_bytes_total{device!~"lo|docker.*|veth.*"}[1m])) by (instance)'
    r = prom_range(expr, hours=0.5, step='15s')
    df3 = to_df(r, 'total_rx')
    print(f"模式 3 — sum by（聚合）：")
    print(f"  所有介面合計平均receive rate：{df3['total_rx'].mean()/1e6:.3f} MB/s")
    print(f"  PromQL：{expr}")
except Exception as e:
    print(f"模式 3 — sum by：⚠ {e}")

In [ ]:
# PromQL 範例 4：label filter — 精確過濾
# 只查詢特定介面，排除 loopback 和 docker 虛擬介面
print("模式 4 — 標籤過濾語法：")
print()
examples = [
    ('=',  'device="eth0"',              '精確匹配'),
    ('!=', 'device!="lo"',               '排除特定值'),
    ('=~', 'device=~"eth.*|en.*"',       '正則匹配（包含）'),
    ('!~', 'device!~"lo|docker.*|veth.*"', '正則排除'),
]
for op, example, desc in examples:
    print(f"  {op:2s}  {example:<35} # {desc}")

print()
print("本 Lab 主要使用：device!~\"lo|docker.*|veth.*\"")
print("  → 排除迴路介面（lo）、Docker 介面（docker0）、虛擬網卡（veth*）")

### PromQL 模式小結

執行完上面 4 個範例後，你已經掌握了 PromQL 的核心語法。

**常見組合查詢（進階應用）：**

```promql
# Find the top 3 interfaces by traffic (past 5 minutes)
topk(3, rate(node_network_receive_bytes_total{device!~"lo|docker.*"}[5m]))

# Calculate error rate (as a percentage)
100 * rate(node_network_receive_errs_total[1m])
  / rate(node_network_receive_packets_total[1m])

# Predict traffic 30 minutes ahead (linear extrapolation)
predict_linear(
  node_network_receive_bytes_total{device="en0"}[30m],
  1800  # predict value 1800 seconds (30 minutes) into the future
)
```

**`predict_linear()` 在 AIOps 中的用途：**
這就是 Lab 08 預測層的 PromQL 版本——用最近趨勢外推未來值，提前預警。
在 Grafana Alert Rule 中搭配 `predict_linear() > 閾值` 即可實現早期預警。


---
**探索練習** — 修改以下 cell 中的參數，觀察結果變化。

---

## 探索：自訂 PromQL 查詢

修改下方的 `QUERY` 和 `HOURS`，查詢不同指標或時間範圍。

**建議嘗試的查詢（由簡到難）：**

```python
# 1. Memory availability (Gauge — use directly)
QUERY = 'node_memory_MemFree_bytes'

# 2. CPU idle fraction
QUERY = 'rate(node_cpu_seconds_total{mode="idle"}[1m])'

# 3. Transmit rate for the past 1 minute (change receive → transmit)
QUERY = 'rate(node_network_transmit_bytes_total{device!~"lo|docker.*|veth.*"}[1m])'

# 4. Total receive rate across all interfaces
QUERY = 'sum(rate(node_network_receive_bytes_total{device!~"lo|docker.*"}[1m]))'

# 5. Receive error rate (as a percentage)
QUERY = '''
100 * rate(node_network_receive_errs_total{device!~"lo|docker.*"}[1m])
  / rate(node_network_receive_packets_total{device!~"lo|docker.*"}[1m])
'''
```

**觀察重點：**
- Gauge 類型（記憶體）vs Counter 類型（流量）的圖形有何不同？
- 加總（sum）後只剩一條線，而不是每個介面各一條
- 錯誤率在流量低時有什麼特性？（提示：小分母問題）


In [ ]:
# 🔧 修改這裡試試看！
QUERY = 'rate(node_network_transmit_bytes_total{device!~"lo|docker.*|veth.*"}[1m])'
HOURS = 1   # 查詢最近幾小時（試試 0.5, 1, 3）
STEP  = "30s"  # 資料間隔（試試 "15s", "1m", "5m"）

try:
    r = prom_range(QUERY, hours=HOURS, step=STEP)
    df_explore = to_df(r, 'tx_rate')
    print(f"✅ 查詢成功：{len(df_explore)} 筆資料")
    print(f"裝置：{df_explore['device'].unique().tolist()}")
    
    fig, ax = plt.subplots(figsize=(12, 3))
    for dev, grp in df_explore.groupby("device"):
        grp = grp.sort_values("timestamp")
        ax.plot(grp["timestamp"], grp["tx_rate"] / 1e6, label=dev)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.set_ylabel("Transmit rate (MB/s)")
    ax.set_title(f"PromQL exploration: {QUERY[:60]}...")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠ {e}")
    print("提示：可以嘗試修改 QUERY 為 'node_memory_MemFree_bytes'")

---
**討論暫停 ▸ 全班討論** — 暫停執行，與全班分享你的觀察。

---

---

## 兩條軌道如何整合：Lab 邏輯 → 自動化監控

今天在 lab 裡驗證的每一個偵測規則，都有對應的部署路徑。

```
Workshop Labs (batch validation)              Deployment to production (reference path)
────────────────────────────────────────      ──────────────────────────────────────────────

Lab 01 Feature Engineering                    Path 1 — Prometheus Recording Rules
  rate(node_network_receive_bytes_total)   →    recording rule: network:rx_rate_1m
  error_rate = rx_errors / rx_packets      →    recording rule: network:error_rate

Lab 02 Anomaly Detection                      Path 1 (cont.) — Alert Rules
  z > 3 → anomaly flag                     →    alert TrafficSurge: expr: z > 3
  change_point flag                        →    alert dual-MA divergence alert rule

Lab 08 Agentic AI RCA                         Path 2 — Webhook + LLM service
  alert → context → Anthropic API          →    alertmanager webhook → rca_service.py
  tool_use → root-cause list               →    → Slack / PagerDuty / ticketing system

⚠ Every arrow requires a human validation checkpoint before entering production
```

| 特性 | 路徑 1（PromQL rules）| 路徑 2（LLM webhook）|
|---|---|---|
| 延遲 | < 5 秒 | 2–15 s (LLM inference time)|
| 適用 | 統計/比率型偵測 | 多維上下文分析、自然語言摘要 |
| 維護成本 | 低（YAML 規則）| 中（服務 + prompt 管理）|
| 可解釋性 | 直接（PromQL 即邏輯）| 需要評估 LLM 輸出 |


## ⚠ 人類驗證點 #0 — 你能看到自己的流量嗎？

### 判斷標準

| 問題 | 預期答案 | 如果不符 |
|------|---------|----------|
| Prometheus 能連線？ | HTTP 200 | 確認 Prometheus 服務是否啟動：`http://localhost:9090` |
| 看到真實網路介面？ | 非 lo/docker | 確認 node_exporter 已啟動並在 prometheus.yml 設定目標 |
| 流量圖有變化？ | 非零且有波動 | 確認網路活動（開個網頁試試）|
| 峰值出現時間合理？ | 符合你的使用行為 | 可能是背景程式在傳輸（OS 更新、雲端同步）|

---

### 決策框架：下一步怎麼做

```
Q: Cannot connect to Prometheus?
   ├─ Yes → Stop here; ask the instructor for help
   └─ No  → Continue

Q: Only seeing lo or docker interfaces?
   ├─ Yes → Adjust the regex filter in the PromQL query
   └─ No  → Continue

Q: Traffic chart is flat (almost no movement)?
   ├─ Yes → Open a few web pages or run `curl` to generate traffic
   └─ No  → Traffic chart is healthy; proceed to Lab 01
```

---

### 討論問題

> 「你的網路介面叫什麼名稱？Linux、macOS、視窗s 的介面名稱有什麼不同？」

> 「你在流量圖中看到任何峰值嗎？那個時間點你在做什麼？」

> 「為什麼我們要用 `rate()` 而不是直接看 `receive_bytes_total` 的原始值？」

> 「如果 scrape_interval 從 15 秒改為 60 秒，流量圖會有什麼變化？對異常偵測有什麼影響？」

---

### 延伸思考：scrape interval 的取捨

| scrape_interval | 時序精度 | 儲存成本 | 適合偵測 |
|-----------------|---------|---------|---------|
| 5s | 最高 | 3× 高 | 毫秒級 SLA |
| 15s（預設）| 高 | 基準 | 一般運維告警 |
| 60s | 中 | 4× 低 | 趨勢分析、容量規劃 |

沒有「最好」的 scrape_interval——它取決於你願意在儲存成本和偵測精度之間做的取捨。


---
**探索練習** — 完成上方討論後，嘗試以下額外練習。

---

## 探索練習 — 查看錯誤率

修改建議：
- 將 `RATE_WINDOW` 從 `"1m"` 改為 `"5m"` — 觀察曲線平滑度變化
- 將指標換為 `node_network_receive_errs_total` — 查看接收錯誤率
- 加入第二條線：同時繪製接收和Transmit速率

In [ ]:
# 🔧 探索：查看receive error rate（修改 RATE_WINDOW 觀察差異）
RATE_WINDOW = "1m"  # 試試 "5m", "15m"

filter_str = 'device!~"lo|docker.*|veth.*"'

try:
    errs = prom_range(
        f'rate(node_network_receive_errs_total{{{filter_str}}}[{RATE_WINDOW}])',
        hours=1, step="15s"
    )
    df_err = to_df(errs, 'err_rate')
    max_err = df_err['err_rate'].max()
    print(f"receive error rate（rate window={RATE_WINDOW}）：")
    print(f"  最大值：{max_err:.4f} errors/sec")
    if max_err < 0.01:
        print("  ✅ 錯誤率很低 — 網路狀態良好")
    else:
        print("  ⚠ 有明顯錯誤 — 可能有網路問題")
except Exception as e:
    print(f"⚠ {e} — 在正常運作的網路中，錯誤率通常接近 0")

print()
print("Lab 00 完成！接下來進入 Lab 01 — 時序特徵工程。")